# `auto_clean`: one function that cleans the whole table
```python
import sys; sys.path.insert(0, "..")
from clean_tools import auto_clean, load_any
df = auto_clean(load_any("file.csv"), target="y_column")
```
It prints every decision, so you can explain it to the teacher:

| Step | What it does |
|---|---|
| 1 | strip spaces in column names |
| 2 | `?`, `NA`, `-`, `null`, `-999`, blank → NaN; text → lowercase, no spaces |
| 3 | drop duplicate rows (`drop_dup=False` to keep) |
| 4 | text that is really numbers (`1,200`, `$50`, `30%`) → numbers |
| 5 | drop rows where the **target** is missing (never fill y) |
| 6 | dates → year / month / day / weekday (+ hour) |
| 7 | drop useless columns: id, names/free text, one single value, > 50 % missing |
| 8 | fill missing: number → median (+ `_missing` 0/1 flag if ≥ 5 % missing), text → mode |
| 9 | text → number: 2 values → 0/1, ≤ 10 values → one-hot, more → frequency |
| + | target as text → 0, 1, 2 … |

In [1]:
import os, sys, warnings
os.environ["LOKY_MAX_CPU_COUNT"] = "4"
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from clean_tools import auto_clean, load_any

## 1. A very dirty table

In [2]:
dirty = pd.DataFrame({
    " ID ":   range(1, 9),
    "Name":   ["Ann", "Bob", "Cat", "Dan", "Eve", "Fin", "Gus", "Hal"],
    "Price":  ["1,200", "$300", "?", "450", "1,000", None, "800", "700"],
    "Date":   ["01/02/2024", "05/03/2024", "10/04/2024", "11/04/2024", "12/05/2024", "01/06/2024", "02/07/2024", "03/08/2024"],
    "City":   [" BKK", "bkk ", "CNX", None, "BKK", "HKT", "CNX", "BKK"],
    "Rooms":  [1, 2, -999, 3, np.nan, 2, 1, 2],
    "Const":  [5] * 8,
    "Sold":   ["yes", "no", "yes", "yes", None, "no", "no", "yes"],
})
dirty

,ID,Name,Price,Date,City,Rooms,Const,Sold
0,1,Ann,"1,200",01/02/2024,BKK,1.0,5,yes
1,2,Bob,$300,05/03/2024,bkk,2.0,5,no
2,3,Cat,?,10/04/2024,CNX,-999.0,5,yes
3,4,Dan,450,11/04/2024,NaN,3.0,5,yes
4,5,Eve,"1,000",12/05/2024,BKK,NaN,5,NaN
5,6,Fin,NaN,01/06/2024,HKT,2.0,5,no
6,7,Gus,800,02/07/2024,CNX,1.0,5,no
7,8,Hal,700,03/08/2024,BKK,2.0,5,yes


In [3]:
clean = auto_clean(dirty, target="Price")
clean

start: 8 rows x 8 cols
  Rooms: -999 -> NaN
  Price: text -> number
  dropped 2 rows with missing target 'Price'
  Date: date -> year/month/day/weekday
  drop ID: id column
  drop Name: text with 6 different values (names / free text)
  drop Const: only one value
  drop Date_year: only one value
  City: 17% missing -> mode 'bkk'
  Rooms: 17% missing -> median 2 (+ _missing flag)
  Sold: 17% missing -> mode 'yes'
  City: 2 values -> 0/1  (bkk=0, cnx=1)
  Sold: 2 values -> 0/1  (no=0, yes=1)
done: 6 rows x 8 cols, missing left = 0


,Price,City,Rooms,Sold,Date_month,Date_day,Date_weekday,Rooms_missing
0,1200.0,0,1.0,1,2,1,3,0
1,300.0,0,2.0,0,3,5,1,0
2,450.0,0,3.0,1,4,11,3,0
3,1000.0,0,2.0,1,5,12,6,1
4,800.0,1,1.0,0,7,2,1,0
5,700.0,0,2.0,1,8,3,5,0


## 2. Titanic (all 12 columns, no hand work)

In [4]:
df = auto_clean(load_any("../titanic.csv"), target="Survived")
df.head()

start: 891 rows x 12 cols
  drop PassengerId: id column
  drop Name: text with 891 different values (names / free text)
  drop Cabin: 77% missing
  Age: 20% missing -> median 28 (+ _missing flag)
  Embarked: 0% missing -> mode 's'
  Sex: 2 values -> 0/1  (female=0, male=1)
  Ticket: 681 values -> frequency (how common each value is)
  Embarked: 3 values -> one-hot (2 cols)
done: 891 rows x 11 cols, missing left = 0


,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Age_missing,Embarked_q,Embarked_s
0,0,3,1,22.0,1,0,0.001122,7.2500,0,0,1
1,1,1,0,38.0,1,0,0.001122,71.2833,0,0,0
2,1,3,0,26.0,0,0,0.001122,7.9250,0,0,1
3,1,1,0,35.0,1,0,0.002245,53.1000,0,0,1
4,0,3,1,35.0,0,0,0.001122,8.0500,0,0,1


## 3. Does it help? Same model, your 3 columns vs. auto_clean

In [5]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import HistGradientBoostingClassifier

raw = pd.read_csv("../titanic.csv")
mine = raw[["Age", "Pclass", "SibSp", "Survived"]].dropna()
m = HistGradientBoostingClassifier(random_state=42)
print("3 columns + dropna :", round(cross_val_score(m, mine.drop(columns="Survived"), mine["Survived"], cv=5).mean(), 4))
print("auto_clean (all)   :", round(cross_val_score(m, df.drop(columns="Survived"), df["Survived"], cv=5).mean(), 4))

3 columns + dropna : 0.7031


auto_clean (all)   : 0.8205


## 4. Use it in Q1 / Q2
```python
df = auto_clean(load_any("../exam.csv"), target="Fare")      # Q1
cols = [c for c in df.columns if c != "Fare"]                 # every clean column as x
```
Q2 has 2 targets: clean with one, then keep the other out of `cols`:
```python
df = auto_clean(load_any("../exam.csv"), target="Survived")
cols = [c for c in df.columns if c not in ["Survived", "Fare"]]
```
Settings you can change: `max_missing=0.5` (drop a column above this), `max_onehot=10`, `drop_dup=False`, `verbose=False`.

> It uses the median of the whole table (before the split). That is normal in class. If the teacher is strict about "no leakage", fill with the train median instead (see `Data_Prep_Examples` section 16).